# Visão Computacional

In [ ]:
import json
import cv2 
import numpy as np 
from pathlib import Path
from ultralytics import YOLO 
import easyocr
import re
import time
from collections import Counter, defaultdict, deque
from IPython.display import Image, clear_output, display

In [ ]:
#Iniciatialize YOLO & OCR
model = YOLO("license_plate_best.pt") # weights
reader = easyocr.Reader(['pt'], gpu=True) 

# Regex: O projeto final prioriza Brasil, mas mantemos como fallback de teste.
old_model_br = re.compile(r"^[A-Z]{3}[0-9]{4}$")
mercosul_model_br = re.compile(r"^[A-Z]{3}[0-9][A-Z][0-9]{2}$")

# O OCR costuma confundir letras e números parecidos.
NUM_TO_ALPHA = {
    "0": "O",
    "1": "I",
    "2": "Z",
    "5": "S",
    "8": "B",
}

ALPHA_TO_NUM = {
    "O": "0",
    "I": "1",
    "Z": "2",
    "S": "5",
    "B": "8",
}


def clean_plate_text(text):
    # Normaliza o texto para só letras maiúsculas e números.
    text = text.upper().replace(" ", "").replace("-", "")
    return re.sub(r"[^A-Z0-9]", "", text)


def correct_plate_format_old(ocr_text):
    # Padrão antigo brasileiro: LLLNNNN
    ocr_text = clean_plate_text(ocr_text)
    if len(ocr_text) != 7:
        return ""

    corrected = []

    for i, ch in enumerate(ocr_text):
        if i < 3:
            if ch.isdigit() and ch in NUM_TO_ALPHA:
                corrected.append(NUM_TO_ALPHA[ch])
            elif ch.isalpha():
                corrected.append(ch)
            else:
                return ""
        else:
            if ch.isdigit():
                corrected.append(ch)
            elif ch.isalpha() and ch in ALPHA_TO_NUM:
                corrected.append(ALPHA_TO_NUM[ch])
            else:
                return ""

    candidate = "".join(corrected)
    return candidate if old_model_br.fullmatch(candidate) else ""


def correct_plate_format_mercosul(ocr_text):
    # Padrão Mercosul brasileiro: LLLNLNN
    ocr_text = clean_plate_text(ocr_text)
    if len(ocr_text) != 7:
        return ""

    corrected = []

    for i, ch in enumerate(ocr_text):
        if i < 3 or i == 4:
            if ch.isdigit() and ch in NUM_TO_ALPHA:
                corrected.append(NUM_TO_ALPHA[ch])
            elif ch.isalpha():
                corrected.append(ch)
            else:
                return ""
        else:
            if ch.isdigit():
                corrected.append(ch)
            elif ch.isalpha() and ch in ALPHA_TO_NUM:
                corrected.append(ALPHA_TO_NUM[ch])
            else:
                return ""

    candidate = "".join(corrected)
    return candidate if mercosul_model_br.fullmatch(candidate) else ""


def empty_plate_info():
    return {
        "plate": "",
        "format": "UNKNOWN",
        "raw_text": "",
        "valid": False,
    }

def recognize_plate(plate_crop):
    if plate_crop.size == 0:
        return empty_plate_info()
    
    # Pré-processamento para melhorar o OCR.
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    plate_resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    best_info = empty_plate_info()

    try: 
        ocr_result = reader.readtext(
            plate_resized, 
            detail=0, 
            allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
        )

        for raw_text in ocr_result:
            cleaned = clean_plate_text(raw_text)
            if not cleaned:
                continue

            candidate = correct_plate_format_old(cleaned)
            if candidate:
                return {
                    "plate": candidate,
                    "format": "BR_OLD",
                    "raw_text": cleaned,
                    "valid": True,
                }

            candidate = correct_plate_format_mercosul(cleaned)
            if candidate:
                return {
                    "plate": candidate,
                    "format": "BR_MERCOSUL",
                    "raw_text": cleaned,
                    "valid": True,
                }

    except: 
        pass
    
    return best_info

plate_history = defaultdict(lambda: deque(maxlen=10)) # last 10
plate_final = {}

def get_box_id(x1, y1, x2, y2):
    # Usa coordenadas arredondadas como pseudo ID.
    return f"{int(x1/10)}_{int(y1/10)}_{int(x2/10)}_{int(y2/10)}"

def get_stable_plate_info(box_id, plate_info):
    if plate_info["valid"]:
        plate_history[box_id].append(plate_info)

        plates = [item["plate"] for item in plate_history[box_id] if item["plate"]]
        if plates:
            most_common_plate = max(set(plates), key=plates.count)

            for item in reversed(plate_history[box_id]):
                if item["plate"] == most_common_plate:
                    plate_final[box_id] = {
                        "plate": most_common_plate,
                        "format": item["format"],
                        "raw_text": item["raw_text"],
                        "valid": True,
                    }
                    break

    return plate_final.get(box_id, empty_plate_info())


def show_frame_in_notebook(frame):
    success, buffer = cv2.imencode(".jpg", frame)
    if not success:
        return

    clear_output(wait=True)
    display(Image(data=buffer.tobytes()))


def build_metrics_summary(detected_results, frame_count, elapsed_time):
    valid_results = [item for item in detected_results if item["valid"]]
    stable_results = [item for item in detected_results if item["stable"]]
    formats = Counter(item["format"] for item in valid_results if item["format"])
    unique_plates = sorted({item["plate"] for item in valid_results if item["plate"]})

    total_detections = len(detected_results)
    fps = frame_count / elapsed_time if elapsed_time > 0 else 0.0

    return {
        "frames_processados": frame_count,
        "tempo_total_segundos": round(elapsed_time, 2),
        "fps_medio": round(fps, 2),
        "deteccoes_totais": total_detections,
        "deteccoes_por_frame": round(total_detections / frame_count, 4) if frame_count else 0.0,
        "placas_unicas_validas": len(unique_plates),
        "taxa_ocr_valido": round(len(valid_results) / total_detections, 4) if total_detections else 0.0,
        "taxa_estabilizacao": round(len(stable_results) / total_detections, 4) if total_detections else 0.0,
        "placas_validas_por_formato": dict(formats),
        "placas_detectadas": unique_plates,
    }


def save_metrics_summary(metrics_summary, output_path="saved_models/inference_metrics.json"):
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(
        json.dumps(metrics_summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return output

# Captura da câmera padrão
input_video = 0

cap = cv2.VideoCapture(input_video) #lê frame ou webcam
if not cap.isOpened():
    raise RuntimeError("Nao foi possivel abrir a camera.")

CONF_THRESH = 0.3 #confiança - voltar aqui para ver com base na regra de negocio
DISPLAY_EVERY_N_FRAMES = 5
detected_results = []
frame_count = 0
start_time = time.time()

# operando frame by frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    frame_results = []

    results = model(frame, verbose=False)

    for r in results: 
        boxes = r.boxes
        for box in boxes:
            conf = float(box.conf.item()) #calculo da confiança
            if conf < CONF_THRESH: 
                continue
    
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            plate_crop = frame[y1:y2, x1:x2]

            # OCR com validação por formato.
            plate_info = recognize_plate(plate_crop)
            text = plate_info["plate"]
            plate_format = plate_info["format"]

            # Estabiliza o resultado usando o histórico da mesma região.
            box_id = get_box_id(x1, y1, x2, y2)
            stable_info = get_stable_plate_info(box_id, plate_info)
            stable_text = stable_info["plate"]
            stable_format = stable_info["format"]
            is_stable = bool(stable_info["valid"])

            result_info = {
                "plate": stable_text or text,
                "format": stable_format if stable_text else plate_format,
                "raw_text": plate_info["raw_text"],
                "valid": stable_info["valid"] or plate_info["valid"],
                "stable": is_stable,
                "confidence": conf,
                "bbox": [x1, y1, x2, y2],
                "timestamp": time.time(),
            }
            frame_results.append(result_info)

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0, 255, 0), 3)

            # Mostra a placa estável e o tipo reconhecido.
            if stable_text:
                label = f"{stable_text} | {stable_format}"
            elif text:
                label = f"{text} | {plate_format}"
            else:
                label = ""

            if label:
                text_y = max(30, y1 - 20)
                cv2.putText(
                    frame, 
                    label, 
                    (x1, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 4
                )
                cv2.putText(
                    frame, 
                    label, 
                    (x1, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2
                )

    if frame_results:
        detected_results.extend(frame_results)
        print(frame_results)

    if frame_count % DISPLAY_EVERY_N_FRAMES == 0:
        show_frame_in_notebook(frame)

elapsed_time = time.time() - start_time

#Close all
cap.release()
show_frame_in_notebook(frame)


## Extração de Métricas

In [ ]:
runtime_elapsed = globals().get("elapsed_time")
processed_frames = globals().get("frame_count", 0)
results_history = globals().get("detected_results", [])

if runtime_elapsed is None:
    if "start_time" in globals():
        runtime_elapsed = max(0.0, time.time() - start_time)
    elif results_history:
        runtime_elapsed = max(0.0, results_history[-1]["timestamp"] - results_history[0]["timestamp"])
    else:
        runtime_elapsed = 0.0

metrics_summary = build_metrics_summary(results_history, processed_frames, runtime_elapsed)
metrics_file = save_metrics_summary(metrics_summary)

print("Resumo das metricas:")
print(json.dumps(metrics_summary, indent=2, ensure_ascii=False))
print(f"Metricas salvas em: {metrics_file}")

NameError: name 'elapsed_time' is not defined